# 02 · Balanced sampling, Braintrust upload & queuing a run

This notebook builds a **deterministic, class-balanced dataset slice** exactly the way
the slice builders do (`scripts/braintrust/create_braintrust_800_dataset.py`):

1. Sample `N` images per class from the committed Braintrust slice.
2. De-duplicate in **rendered-pixel space** (hash the normalized PNG, never raw bytes).
3. Normalize each image to a grayscale 1024x1024 PNG and show the exact upload payload.
4. Show the eval-run command the slice builders queue (preview only — no credits spent).

To keep this site build clean, the upload and eval steps are **print-only previews**;
set `RUN_UPLOAD = True` (or run the printed commands) to actually create the slice and
queue a run.

## 0. Bootstrap: repo path + credentials

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src" / "constants.py").exists()
)
sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)


In [ ]:
from src.braintrust_config import load_braintrust_config
from src.env_utils import require_env

config = load_braintrust_config()      # braintrust.env first, then .env
api_key = require_env("OPENROUTER_API_KEY")[0]

print("project:", config.project_name)
print("project_id:", config.project_id)
print("dataset:", config.dataset_project, "/", config.dataset)
print("model:", config.model)
print("braintrust api_key set:", bool(config.api_key))
print("openrouter api_key set:", bool(api_key))


## 1. Sampling parameters

`N_PER_CLASS` and `SEED` are your spec: the same seed + source always reproduces the
same slice. The slice has `16 * N_PER_CLASS` rows (each of the 16 classes in
`src.constants.DOCUMENT_CLASSES`).

In [ ]:
from src.constants import DOCUMENT_CLASSES

N_PER_CLASS = 2   # EDIT: images per class -> 16 * N_PER_CLASS total rows
SEED = 42         # EDIT: deterministic sampling seed
print(f"{N_PER_CLASS} per class x {len(DOCUMENT_CLASSES)} classes = {N_PER_CLASS * len(DOCUMENT_CLASSES)} rows")


## 2. Balanced sample from the committed Braintrust slice

The committed slice is already deterministic and pixel-deduped, so drawing the first
`N_PER_CLASS` rows per class reproduces a stable sample with no local RVL-CDIP tree
required. Each row's PNG attachment is downloaded directly (`fetch_attachment_bytes`),
mirroring how the production slice builders pull images.

In [ ]:
from io import BytesIO

import braintrust

from src.braintrust_utils import fetch_attachment_bytes


def sample_balanced_from_braintrust(n_per_class: int):
    braintrust.login(api_key=config.api_key)
    ds = braintrust.init_dataset(project=config.project_name, name=config.dataset)
    counts = {cls: 0 for cls in DOCUMENT_CLASSES}
    selected: list[tuple[str, bytes, str]] = []
    for row in ds:
        if all(counts[cls] >= n_per_class for cls in DOCUMENT_CLASSES):
            break
        meta = (row.get("input") or {}).get("metadata") or {}
        att = (row.get("input") or {}).get("image")
        if meta.get("placeholder") or att is None:
            continue
        cls = row.get("expected")
        if cls not in counts or counts[cls] >= n_per_class:
            continue
        counts[cls] += 1
        raw = fetch_attachment_bytes(config.api_key, att.reference, config.org_id, config.api_base)
        fn = att.reference.get("filename") or f"rvl_cdip__{cls}__{counts[cls]:04d}.png"
        selected.append((cls, raw, fn))
    return selected


selected = sample_balanced_from_braintrust(N_PER_CLASS)
print(f"{len(selected)} images sampled ({N_PER_CLASS}/class x {len(DOCUMENT_CLASSES)} classes)")
for cls, raw, fn in selected[:6]:
    print(" ", cls, fn, f"({len(raw):,} bytes)")


## 3. Normalize + pixel-hash de-duplication

De-duplication is enforced on the **normalized rendered PNG**, so identical images from
different files cannot slip past. Each image is rendered to a grayscale 1024x1024 PNG
first, hashed, and skipped if the hash was already accepted.

In [ ]:
import hashlib
from io import BytesIO

from PIL import Image

from src.image_utils import resize_with_padding


def to_png_bytes(img: Image.Image, target_size=(1024, 1024)) -> bytes:
    img = img.convert("L")
    img = resize_with_padding(img, target_size, fill=255)
    buf = BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def pixel_hash(png_bytes: bytes) -> str:
    return hashlib.sha256(png_bytes).hexdigest()


cls, raw, fn = selected[0]
png = to_png_bytes(Image.open(BytesIO(raw)))
print(cls, fn, "->", len(png), "bytes, hash", pixel_hash(png)[:16])


## 4. Prepare the upload payload (preview only)

The upload matches the documented pattern used by every slice builder:
`braintrust.login` -> `init_dataset` -> `insert` with an `Attachment` payload -> `flush`/`close`. An existing dataset with the same name is deleted first so re-runs
are safe. This demo **builds the exact row payloads** (real normalized PNGs + pixel
hashes) but keeps `RUN_UPLOAD = False` so re-running the notebook never creates junk
datasets in Braintrust.

In [ ]:
from io import BytesIO

from PIL import Image

# Build the exact row payloads the slice builders upload (no network calls).
used: set[str] = set()
rows: list[tuple[str, bytes, str]] = []
for cls, raw, _src_fn in selected:
    png = to_png_bytes(Image.open(BytesIO(raw)))
    h = pixel_hash(png)
    if h in used:
        continue
    used.add(h)
    rows.append((cls, png, f"rvl_cdip__{cls}__{len(rows) + 1:04d}.png"))

print(f"{len(rows)} unique normalized images ready (after pixel-hash dedup)")

# --- Preview of the upload. Set RUN_UPLOAD = True to actually create the   ---
# --- dataset in Braintrust (idempotent: a same-named dataset is deleted    ---
# --- first, so re-runs are safe).                                          ---
RUN_UPLOAD = False
if RUN_UPLOAD:
    import braintrust
    from braintrust import Attachment

    from src.braintrust_utils import delete_dataset_by_name

    dataset_name = f"notebook_balanced_{N_PER_CLASS}"
    braintrust.login(api_key=config.api_key)
    delete_dataset_by_name(config.api_key, config.project_id, dataset_name, config.api_base)
    dataset = braintrust.init_dataset(project_id=config.project_id, name=dataset_name)
    for cls, png, fn in rows:
        dataset.insert(
            input={
                "image": Attachment(data=png, filename=fn, content_type="image/png"),
                "metadata": {"class": cls, "placeholder": False},
            },
            expected=cls,
            metadata={"source": "notebook-balanced-sample", "seed": SEED, "n_per_class": N_PER_CLASS},
        )
    dataset.flush()
    dataset.close()
    print(f"Uploaded {len(rows)} unique images -> dataset '{dataset_name}'")
else:
    print("Upload skipped (RUN_UPLOAD = False). Set RUN_UPLOAD = True to upload "
          f"{N_PER_CLASS * len(DOCUMENT_CLASSES)} rows to Braintrust.")


## 5. Queue an eval run against the slice (preview only)

Preflight validates prompt + dataset (**spends no credits**), then the eval runner
(`braintrust_openrouter_input.py`) executes the slice and streams results into both
Braintrust and a local JSONL manifest. That second step spends OpenRouter credits
(~$0.0004/image), so this demo only prints the commands.

In [ ]:
# Preview of the slice-builder workflow. Executing the eval spends OpenRouter
# credits, so this demo only prints the exact commands to run.
dataset_name = f"notebook_balanced_{N_PER_CLASS}"
experiment_name = f"notebook_{config.model.replace('/', '_')}_v17.2_{dataset_name}"

print("$ python scripts/braintrust/preflight_eval.py \\")
print(f"      --dataset {dataset_name} --prompt-version v17.2")
print()
print("$ python scripts/braintrust/braintrust_openrouter_input.py \\")
print(f"      --dataset {dataset_name} --prompt-version v17.2 \\")
print(f"      --model {config.model} --experiment-name {experiment_name} \\")
print(f"      --manifest reports/manifests/{experiment_name}.jsonl")


## Next

Notebook **03 · Watchers, evaluators & full experiment launch** covers preflight,
monitoring a run from the manifest, crash-proof resume, and the post-run scoring/
reporting chain.